# Sesión 17 — Transformers en Biomedicina
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo V · Arquitecturas Avanzadas y AI Generativa**

## Objetivos de aprendizaje

1. Comprender el paradigma de preentrenamiento de BERT (MLM + NSP) y cómo se adapta al texto clínico.
2. Ajustar finamente (fine-tune) un modelo BERT clínico para predicción de códigos ICD a partir de resúmenes de alta.
3. Comprender los modelos de lenguaje biomédicos específicos de dominio (BioBERT, ClinicalBERT, PubMedBERT).
4. Aplicar un ViT a imágenes médicas y comprender por qué importa el tamaño de parche.
5. Revisar arquitecturas Transformer específicas para EEG (BENDR, LaBraM) y sus estrategias de preentrenamiento.

## Lecturas recomendadas

| Prioridad | Referencia |
|---|---|
| ★★★ | Devlin, J. et al. (2019). BERT: Pre-training of deep bidirectional transformers for language understanding. *NAACL*. |
| ★★★ | Alsentzer, E. et al. (2019). Publicly available clinical BERT embeddings. *ACL Clinical NLP Workshop*. |
| ★★☆ | Lee, J. et al. (2020). BioBERT: a pre-trained biomedical language representation model. *Bioinformatics*, 36(4). |
| ★★☆ | Jiang, Z. et al. (2023). Large language models for clinical NLP: a systematic review. *J. Biomed. Informatics*. |
| ★★☆ | Kostas, D. et al. (2022). BENDR: EEG foundation model using contrastive self-supervised learning. |
| ★☆☆ | HuggingFace transformers: https://huggingface.co/docs/transformers |

## Parte 0 — Configuración

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.model_selection import train_test_split

rng    = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
})

# Verificar disponibilidad de HuggingFace transformers
try:
    from transformers import (AutoTokenizer, AutoModel,
                               AutoModelForSequenceClassification,
                               TrainingArguments, Trainer)
    HAS_HF = True
    print('HuggingFace transformers disponible.')
except ImportError:
    HAS_HF = False
    print('transformers no instalado. Ejecutar: pip install transformers datasets')
    print('Usando demostración ligera alternativa.')

print(f'Dispositivo: {device}')

## Parte 1 — Objetivos de preentrenamiento de BERT

BERT se preentrena con dos objetivos:

1. **Modelado de Lenguaje Enmascarado (MLM):** se enmascara el 15% de los tokens;
   el modelo los predice a partir del contexto bidireccional.
2. **Predicción de la Siguiente Oración (NSP):** dadas dos oraciones, predecir si
   la segunda sigue a la primera.

Ajuste fino: anteponer `[CLS]` → encoder → clasificar la representación del CLS.

In [ ]:
# ── Ilustrar la estrategia de enmascaramiento de MLM ──────────────────────────
texto_clinico = (
    "El paciente presentó dolor torácico de inicio agudo irradiado al brazo izquierdo. "
    "El ECG mostró elevación del ST en derivaciones V1-V4 consistente con STEMI anterior. "
    "La troponina I estaba elevada a 8.4 ng/mL. El paciente fue llevado urgentemente al laboratorio de cateterismo."
)

tokens = texto_clinico.split()
n_tokens = len(tokens)
mask_prob = 0.15

rng_mask = np.random.default_rng(7)
tokens_enmascarados = tokens.copy()
indices_enmascarados = []

for i, tok in enumerate(tokens):
    if rng_mask.random() < mask_prob:
        r = rng_mask.random()
        indices_enmascarados.append(i)
        if r < 0.80:              # 80%: reemplazar con [MASK]
            tokens_enmascarados[i] = '[MASK]'
        elif r < 0.90:            # 10%: reemplazar con token aleatorio
            tokens_enmascarados[i] = rng_mask.choice(tokens)
        # si no: 10%: mantener el original

print('=== Demostración de Modelado de Lenguaje Enmascarado (MLM) de BERT ===\n')
print('Original:    ', ' '.join(tokens))
print()
print('Enmascarado: ', ' '.join(tokens_enmascarados))
print()
print(f'Enmascarados {len(indices_enmascarados)}/{n_tokens} tokens '
      f'({100*len(indices_enmascarados)/n_tokens:.1f}%)')
print('Posiciones enmascaradas:', [tokens[i] for i in indices_enmascarados])

# Visualizar
fig, ax = plt.subplots(figsize=(14, 2.5))
for i, (orig, masked) in enumerate(zip(tokens, tokens_enmascarados)):
    color = 'tomato' if i in indices_enmascarados else 'steelblue'
    ax.text(i * 0.95, 0.6, orig,   fontsize=7.5, ha='center', color='gray')
    ax.text(i * 0.95, 0.2, masked, fontsize=7.5, ha='center', color=color,
            fontweight='bold' if i in indices_enmascarados else 'normal')
ax.text(-1, 0.6, 'Original:',    ha='right', fontsize=9, color='gray')
ax.text(-1, 0.2, 'Enmascarado:', ha='right', fontsize=9, color='black')
ax.set(xlim=(-2, len(tokens)*0.95), ylim=(0, 1))
ax.axis('off')
ax.set_title('Enmascaramiento MLM — tokens en rojo están enmascarados; '
              'el modelo debe predecir sus valores originales', pad=8)
plt.tight_layout()
plt.show()

## Parte 2 — Ajuste fino de BERT clínico para predicción de código ICD

Simulamos una tarea de clasificación de resúmenes de alta: dado el resumen, predecir
si se asigna el código ICD-10 **I21** (infarto agudo de miocardio).

In [ ]:
# ── Simular resúmenes clínicos de alta ────────────────────────────────────────
plantillas_ami = [
    "Paciente presentó dolor torácico subesternal, diaforesis y disnea. "
    "ECG demostró elevación del ST. Troponina elevada. Sometido a ICP primaria con colocación de stent.",
    "Inicio agudo de opresión torácica severa irradiada a mandíbula y brazo izquierdo. "
    "Cateterismo urgente reveló oclusión del 95% de la DA. Se colocó stent liberador de fármaco.",
    "Hombre de 64 años con dolor torácico opresivo, elevación del ST en derivaciones anteriores V1-V4. "
    "Tratado con trombolisis seguida de ICP. Fracción de eyección reducida a 35%.",
    "STEMI inferior confirmado con elevación del ST en II, III, aVF. ACD totalmente ocluida. "
    "Revascularización exitosa. Egresado con doble terapia antiplaquetaria.",
    "Presentación de NSTEMI: dolor torácico, troponina elevada, cambios dinámicos del ST. "
    "Manejo conservador con anticoagulación y angiografía temprana realizada.",
]

plantillas_control = [
    "Paciente ingresado con neumonía, tos productiva, fiebre de 38.9°C. "
    "Radiografía de tórax mostró consolidación del lóbulo inferior derecho. Tratado con amoxicilina.",
    "Diabetes mellitus tipo 2 de inicio reciente. HbA1c 9.2%. Se inició terapia con metformina. "
    "Se brindó orientación dietética. Se programó seguimiento.",
    "Cirugía electiva de reemplazo de cadera. Sin antecedentes cardíacos significativos. "
    "Curso posoperatorio sin complicaciones. Fisioterapia iniciada el primer día.",
    "Ingresado por urgencia hipertensiva. PA 190/120 mmHg. "
    "Se inició amlodipino y ramipril. Egresado con monitoreo de PA de 24h.",
    "Apendicitis aguda confirmada por TC. Apendicectomía laparoscópica realizada. "
    "Histología confirmó inflamación aguda. Sin perforación.",
]

# Generar dataset con variación
n_samples = 400
notas, etiquetas = [], []

for _ in range(n_samples // 2):
    base = rng.choice(plantillas_ami)
    # Añadir ruido: prefijo aleatorio
    prefijo = rng.choice([
        'Diagnóstico de ingreso: IAM agudo. ',
        'Resumen de alta: ',
        'Resumen: ',
        '',
    ])
    notas.append(prefijo + base)
    etiquetas.append(1)

for _ in range(n_samples // 2):
    base = rng.choice(plantillas_control)
    notas.append(base)
    etiquetas.append(0)

# Mezclar
perm = rng.permutation(len(notas))
notas     = [notas[i]     for i in perm]
etiquetas = [etiquetas[i] for i in perm]

print(f'Dataset: {len(notas)} notas, {sum(etiquetas)} IAM / {len(etiquetas)-sum(etiquetas)} control')
print('\nNota IAM de ejemplo (truncada):')
print(' ', notas[etiquetas.index(1)][:120], '...')

In [ ]:
if HAS_HF:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    from torch.utils.data import Dataset

    MODEL_NAME = 'emilyalsentzer/Bio_ClinicalBERT'
    print(f'Cargando tokenizador: {MODEL_NAME}')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    class ClinicalNoteDataset(torch.utils.data.Dataset):
        def __init__(self, texts, labels, tokenizer, max_len=128):
            self.encodings = tokenizer(texts, truncation=True, padding='max_length',
                                        max_length=max_len, return_tensors='pt')
            self.labels = torch.tensor(labels, dtype=torch.long)

        def __len__(self):  return len(self.labels)
        def __getitem__(self, i):
            return {k: v[i] for k, v in self.encodings.items()}, self.labels[i]

    n_tr = int(0.75 * len(notas))
    train_ds = ClinicalNoteDataset(notas[:n_tr],  etiquetas[:n_tr],  tokenizer)
    test_ds  = ClinicalNoteDataset(notas[n_tr:],  etiquetas[n_tr:],  tokenizer)

    model_bert = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2).to(device)

    # Congelar todo excepto las últimas 2 capas Transformer + clasificador
    for name, param in model_bert.named_parameters():
        if 'classifier' not in name and 'layer.11' not in name and 'layer.10' not in name:
            param.requires_grad = False

    entrenables = sum(p.numel() for p in model_bert.parameters() if p.requires_grad)
    total       = sum(p.numel() for p in model_bert.parameters())
    print(f'Parámetros: {total:,} totales, {entrenables:,} entrenables ({100*entrenables/total:.1f}%)')

    # Ajuste fino
    tr_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    te_loader = DataLoader(test_ds,  batch_size=32)
    opt_bert  = optim.AdamW(filter(lambda p: p.requires_grad, model_bert.parameters()),
                             lr=2e-5, weight_decay=0.01)
    sched_b   = optim.lr_scheduler.LinearLR(opt_bert, start_factor=1.0,
                                             end_factor=0.1, total_iters=10)

    print('\nAjustando finamente ClinicalBERT...')
    for ep in range(5):
        model_bert.train()
        ep_loss = 0
        for batch, ylabels in tr_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            ylabels = ylabels.to(device)
            opt_bert.zero_grad()
            out = model_bert(**batch, labels=ylabels)
            out.loss.backward()
            nn.utils.clip_grad_norm_(model_bert.parameters(), 1.0)
            opt_bert.step()
            ep_loss += out.loss.item()
        sched_b.step()

        model_bert.eval()
        all_probs, all_labs = [], []
        with torch.no_grad():
            for batch, ylabels in te_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                logits = model_bert(**batch).logits
                probs  = F.softmax(logits, dim=1)[:,1].cpu().numpy()
                all_probs.extend(probs)
                all_labs.extend(ylabels.numpy())
        auroc = roc_auc_score(all_labs, all_probs)
        print(f'Época {ep+1}  pérdida={ep_loss/len(tr_loader):.4f}  AUROC={auroc:.4f}')

else:
    # ── Línea base ligera con TF-IDF + regresión logística ────────────────────
    print('Usando línea base TF-IDF + Regresión Logística (sin HuggingFace).')
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline

    n_tr = int(0.75 * len(notas))
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000)),
        ('clf',   LogisticRegression(C=1.0, max_iter=500))
    ])
    pipe.fit(notas[:n_tr], etiquetas[:n_tr])
    probs_tfidf = pipe.predict_proba(notas[n_tr:])[:,1]
    auroc_tfidf = roc_auc_score(etiquetas[n_tr:], probs_tfidf)
    print(f'\nAUROC línea base TF-IDF + LR: {auroc_tfidf:.4f}')
    print('Instala transformers (pip install transformers) para el ajuste fino de ClinicalBERT.')

## Parte 3 — Panorama de modelos de lenguaje biomédicos

| Modelo | Corpus de preentrenamiento | Fortaleza clave |
|---|---|---|
| **BioBERT** | Resúmenes PubMed + texto completo PMC | NER biomédico, extracción de relaciones |
| **ClinicalBERT** | Notas clínicas MIMIC-III | Texto clínico, resúmenes de alta |
| **PubMedBERT** | Solo PubMed (sin Wikipedia) | Vocabulario de dominio más fuerte |
| **GatorTron** | 90 mil millones de palabras de texto clínico | QA clínico a gran escala |
| **Med-PaLM 2** | RLHF en preguntas y respuestas médicas | Razonamiento clínico de nivel experto |
| **LLaMA-Med** | LLaMA abierto + instrucción médica | Razonamiento clínico de código abierto |

**Consideraciones críticas para el NLP clínico:**
- Desidentificación antes del entrenamiento/ajuste fino (PHI bajo HIPAA)
- Desplazamiento temporal de conceptos (el lenguaje clínico evoluciona)
- Riesgo de alucinación en LLM — nunca confiar sin verificación
- Necesidad de anclaje a ontologías médicas (SNOMED-CT, ICD, RxNorm)

In [ ]:
# ── Visualización del espacio de embeddings: BERT clínico vs general ──────────
# Simulamos embeddings de tokens para vocabulario médico vs general

terminos_medicos = [
    'troponina', 'miocárdico', 'infarto', 'ecocardiograma', 'stemi',
    'percutánea', 'angioplastia', 'arritmia', 'bradicardia', 'taquicardia',
    'electroencefalograma', 'ictal', 'postictal', 'crisis', 'epilepsia'
]
terminos_generales = [
    'dolor', 'corazón', 'ataque', 'sangre', 'presión', 'médico', 'hospital',
    'tratamiento', 'paciente', 'medicina', 'examen', 'resultado', 'normal', 'alto', 'bajo'
]

if HAS_HF:
    from transformers import AutoTokenizer, AutoModel
    from sklearn.decomposition import PCA

    todos_terminos = terminos_medicos + terminos_generales
    for model_name, etiqueta in [
        ('bert-base-uncased',              'BERT General'),
        ('emilyalsentzer/Bio_ClinicalBERT','ClinicalBERT'),
    ]:
        tok_v = AutoTokenizer.from_pretrained(model_name)
        mdl_v = AutoModel.from_pretrained(model_name)
        mdl_v.eval()
        embeddings = []
        with torch.no_grad():
            for term in todos_terminos:
                inp = tok_v(term, return_tensors='pt')
                out = mdl_v(**inp)
                # Usar el embedding del token [CLS]
                embeddings.append(out.last_hidden_state[0, 0].numpy())
        embeddings = np.array(embeddings)

        pca = PCA(n_components=2)
        emb_2d = pca.fit_transform(embeddings)

        fig, ax = plt.subplots(figsize=(9, 6))
        ax.scatter(emb_2d[:15, 0], emb_2d[:15, 1], c='tomato',    s=80,
                    label='Términos médicos', zorder=3)
        ax.scatter(emb_2d[15:, 0], emb_2d[15:, 1], c='steelblue', s=80,
                    label='Términos generales', zorder=3)
        for i, term in enumerate(todos_terminos):
            ax.annotate(term, emb_2d[i], fontsize=7.5, alpha=0.8,
                         xytext=(4, 4), textcoords='offset points')
        ax.set(title=f'{etiqueta} — PCA de embeddings de tokens\n'
                      'Los modelos específicos de dominio agrupan mejor los términos clínicos',
               xlabel='PC 1', ylabel='PC 2')
        ax.legend()
        plt.tight_layout()
        plt.show()
else:
    print('HuggingFace no disponible — se omite la visualización de embeddings.')
    print('Instalar con: pip install transformers')

## Parte 4 — Modelos fundacionales de EEG: arquitectura BENDR

BENDR (Kostas et al. 2022) aplica **aprendizaje autosupervisado contrastivo**
(inspirado en wav2vec 2.0) a EEG crudo:

1. Codificar el EEG crudo mediante un **encoder convolucional** → representaciones latentes
2. Cuantizar las representaciones en un **codebook** ("tokens" discretos de EEG)
3. Aplicar **enmascaramiento** a algunas posiciones latentes
4. Una **red de contexto Transformer** debe predecir las posiciones enmascaradas desde el codebook

Esto es esencialmente **BERT para EEG** — no se necesitan etiquetas para el preentrenamiento.

In [ ]:
class BENDREncoder(nn.Module):
    """
    Encoder convolucional simplificado estilo BENDR.
    Mapea EEG crudo (canales × tiempo) → secuencia latente (T', d_model).
    """
    def __init__(self, n_channels=20, d_model=256):
        super().__init__()
        self.temporal = nn.Sequential(
            # Convoluciones temporales con dilatación creciente
            nn.Conv1d(n_channels, 512, kernel_size=3, padding=1),
            nn.GELU(), nn.GroupNorm(32, 512),
            nn.Conv1d(512, 512, kernel_size=3, padding=2, dilation=2),
            nn.GELU(), nn.GroupNorm(32, 512),
            nn.Conv1d(512, 512, kernel_size=3, padding=4, dilation=4),
            nn.GELU(), nn.GroupNorm(32, 512),
            nn.Conv1d(512, d_model, kernel_size=1),   # proyectar a d_model
        )

    def forward(self, x):
        # x: (B, C, T)
        return self.temporal(x).transpose(1, 2)   # → (B, T, d_model)


class BENDRMasker(nn.Module):
    """Aplica enmascaramiento de tramos a una secuencia latente (estilo BERT para EEG)."""
    def __init__(self, mask_ratio=0.065, span_len=10, d_model=256):
        super().__init__()
        self.mask_ratio = mask_ratio
        self.span_len   = span_len
        self.mask_embed = nn.Parameter(torch.randn(d_model))

    def forward(self, x):
        B, T, D = x.shape
        mask = torch.zeros(B, T, dtype=torch.bool, device=x.device)

        for b in range(B):
            n_spans = max(1, int(T * self.mask_ratio))
            for _ in range(n_spans):
                start = torch.randint(0, T - self.span_len, (1,)).item()
                mask[b, start:start + self.span_len] = True

        x_masked = x.clone()
        x_masked[mask] = self.mask_embed
        return x_masked, mask


# Demo: encoder + enmascarador
encoder = BENDREncoder(n_channels=20, d_model=256).to(device)
masker  = BENDRMasker(d_model=256).to(device)

# Simular EEG de 20 canales, 4 segundos a 250 Hz
eeg_raw = torch.randn(4, 20, 1000).to(device)   # (B, C, T)

z = encoder(eeg_raw)           # (B, T', 256)
z_masked, mask = masker(z)

print('Arquitectura estilo BENDR:')
print(f'  EEG crudo de entrada:  {tuple(eeg_raw.shape)}  (batch, canales, tiempo)')
print(f'  Latentes codificados:  {tuple(z.shape)}         (batch, T\', d_model)')
print(f'  Latentes enmascarados: {tuple(z_masked.shape)}')
print(f'  Densidad de enmascaramiento: {mask.float().mean():.3f} de las posiciones')

# Visualizar el enmascaramiento
fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)

axes[0].imshow(z[0].detach().cpu().T[:32, :200],
               aspect='auto', cmap='RdBu_r')
axes[0].set(ylabel='Dim. latente (primeras 32)', title='Latentes de EEG codificados (sin enmascarar)')

z_vis = z_masked[0].detach().cpu().T[:32, :200].numpy()
axes[1].imshow(z_vis, aspect='auto', cmap='RdBu_r')
# Superponer la máscara
mask_vis = mask[0, :200].cpu().numpy()
for t, m in enumerate(mask_vis):
    if m:
        axes[1].axvspan(t-0.5, t+0.5, alpha=0.4, color='yellow')
axes[1].set(xlabel='Paso de tiempo', ylabel='Dim. latente (primeras 32)',
            title='Latentes enmascarados (amarillo = posiciones enmascaradas — el modelo debe predecirlas)')

plt.tight_layout()
plt.show()

## Parte 5 — Estrategias de ajuste fino y consideraciones prácticas

In [ ]:
# Demostrar el beneficio de eficiencia de datos de representaciones preentrenadas

def simular_curva_ajuste_fino(preentrenado=True, n_max=500, seed=0):
    """Simula AUROC vs tamaño del conjunto de entrenamiento etiquetado."""
    rng_ft = np.random.default_rng(seed)
    tamanos = [10, 25, 50, 100, 200, 350, 500]
    aurocs  = []
    for n in tamanos:
        n = min(n, n_max)
        if preentrenado:
            # Modelo preentrenado: alto rendimiento incluso con pocas etiquetas
            base = 0.92 * (1 - np.exp(-n / 40)) + rng_ft.normal(0, 0.02)
        else:
            # Desde cero: necesita muchas más etiquetas
            base = 0.88 * (1 - np.exp(-n / 200)) + rng_ft.normal(0, 0.025)
        aurocs.append(np.clip(base, 0.5, 0.98))
    return tamanos, aurocs

tam_pre, auroc_pre = simular_curva_ajuste_fino(preentrenado=True)
tam_scr, auroc_scr = simular_curva_ajuste_fino(preentrenado=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(tam_pre, auroc_pre, 'b-o', lw=2.5, ms=8,
        label='Preentrenado (ej. ClinicalBERT / BENDR)')
ax.plot(tam_scr, auroc_scr, 'r-s', lw=2.5, ms=8,
        label='Entrenado desde cero')
ax.axhline(0.85, color='gray', ls=':', lw=1, label='Umbral de utilidad clínica (≈0.85)')
ax.fill_between([10, 100], [0.5, 0.5], [1.0, 1.0], alpha=0.06, color='green',
                label='Régimen de pocos datos')
ax.set(xlabel='Ejemplos de entrenamiento etiquetados (escala log)',
       ylabel='AUROC', xscale='log',
       title='Eficiencia de datos: preentrenado vs desde cero\n'
             'Los modelos fundacionales son especialmente valiosos cuando las etiquetas son escasas',
       ylim=(0.45, 1.02))
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print('Idea clave: en contextos clínicos/biomédicos, los datos etiquetados son costosos.')
print('Los modelos fundacionales preentrenados alcanzan utilidad clínica con 5-10× menos etiquetas.')

## ✏️ Ejercicios

1. **NER con BioBERT.** Usando HuggingFace, carga `dmis-lab/biobert-base-cased-v1.2` y
   ajústalo finamente en el dataset de NER de químicos/enfermedades BC5CDR (disponible
   vía la librería `datasets`). Reporta el F1 a nivel de entidad para químicos y
   enfermedades por separado. Compara con una línea base BiLSTM-CRF.

2. **Ingeniería de prompts vs ajuste fino.** Usando un LLM abierto pequeño (ej.
   `microsoft/BiomedNLP-PubMedBERT-base-uncased`), compara: (a) clasificación zero-shot
   con un prompt cuidadosamente diseñado, (b) few-shot (5 ejemplos), y (c) ajuste fino
   completo. Grafica AUROC vs esfuerzo de anotación.

3. **Ablación del tamaño de parche en ViT.** Entrena el ViT-EEG de la Sesión 16 en
   espectrogramas de EEG con tamaños de parche ∈ {4×4, 8×8, 16×16, 32×32}. Grafica
   el AUROC en test y el tiempo de entrenamiento vs el tamaño de parche. Explica el
   compromiso entre detalle local y longitud de secuencia.

4. **Simulación de ajuste fino de BENDR.** Implementa el pipeline completo de BENDR:
   (a) preentrenar el encoder+Transformer en EEG sin etiquetar con pérdida de predicción
   enmascarada, (b) ajustar finamente en datos de crisis etiquetados con cantidades
   variables de etiquetas. Grafica AUROC vs tamaño del conjunto etiquetado (10, 50, 100,
   500 ejemplos).

5. *(Desafío)* **Auditoría de alucinaciones de LLM.** Genera 20 viñetas clínicas y pide
   a un LLM abierto pequeño que sugiera códigos ICD-10. Verifica manualmente cada uno
   contra la verdad de referencia. Categoriza los errores por tipo (familia de código
   incorrecta, plausible pero incorrecto, factualmente incorrecto). Construye un sistema
   de generación aumentada por recuperación (RAG) usando FAISS que ancle las respuestas
   a un codebook ICD-10 local, y mide la reducción de errores.

## 📚 Conjuntos de datos

| Conjunto de datos | Fuente | Notas |
|---|---|
| MIMIC-III / MIMIC-IV | https://physionet.org/content/mimiciii/ | Notas clínicas + códigos ICD |
| BC5CDR NER | librería `datasets`: `bigbio/bc5cdr` | NER de químicos/enfermedades |
| MedNLI | https://physionet.org/content/mednli/ | Inferencia textual clínica |
| Pesos preentrenados BENDR | https://github.com/SPOClab-ca/BENDR | Modelo fundacional de EEG |